In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# ------------------------------------------------------------------------------------------------ #
# Author: Maurice Roots
# Date: 2026-06-18
# Description: Plots for Anne's paper
# ------------------------------------------------------------------------------------------------ #
#%%

In [2]:
# Housekeeping 
from pathlib import Path
import pickle
import copy
import json

# Data Processing
import pandas as pd
import numpy as np

# Geospatial
import geopandas as gpd
from shapely.geometry import Point

# Math and Stats
from scipy.stats import linregress

# Plotting
import matplotlib.pyplot as plt 



# Custom Modules
from atmoz.resources.path_manager import PathManager
import atmoz.resources.importData as importData
import atmoz.resources.geoSlicing as geoSlicing
import atmoz.resources.makePlots as makePlots

### Importing the Datasets from Pickle Files

In [3]:
with open("./paths.json", "r") as f: 
    paths = json.load(f)

data_dir = Path(paths["Laptop_Linux"]["path_datasets"])
fig_dir = Path(paths["Laptop_Linux"]["path_figures"])

data = importData.main_import(data_dir)
tolnet = data['tolnet'].copy()
sondes = data['sondes'].copy()


In [4]:
instrument_names = {
    "csl001_hires_guilford": "NOAA CSL Lidar",
    "gsfc003_hires_oldfield": "NASA GSFC Lidar",
    "gsfc003_hires": "NASA GSFC Lidar",
    'csl001_hires_boulder': "NOAA CSL Lidar",
    "york": "CCNY Lidar",
    "larc001_hires_sherwood": "NASA LaRC Lidar"
}

instrument_locations = {
    "csl001_hires_guilford": "Yale Coastal",
    "gsfc003_hires_oldfield": "Flax Pond",
    "gsfc003_hires": "Flax Pond",
    'csl001_hires_boulder': "Boulder, CO",
    "york": "CCNY Rooftop",
    "larc001_hires_sherwood": "LaRC Sherwood"
}

### Organizing the Data to Match Sondes within 10km of Lidar Sites

In [5]:
# Create TOLNet vs Sonde Comparison
if (data_dir / "comparison.pkl").is_file(): 
    with open(data_dir / "comparison.pkl", "rb") as f:
        comparison = pickle.load(f)
else:
    comparison = {} 
    for tolnet_key in tolnet.keys(): 
        for sonde_key in sondes.index.get_level_values('filename').unique():
            sondes_masked = geoSlicing._add_distance_from_point(sondes.loc[sonde_key, :], location=tolnet[tolnet_key].geometry.unique()[0])
            temp = makePlots._match_time(sondes_masked[sondes_masked["distance_km"] <= 10].drop(columns="geometry"), tolnet[tolnet_key].drop(columns="geometry"))
            if not temp["left_df"].empty and not temp["right_df"].empty: 
                comparison[(tolnet_key, sonde_key)] = temp

    with open(data_dir / "comparison.pkl", "wb") as f:
        pickle.dump(comparison, f)

In [6]:
filenames = [key for key in tolnet.keys() if "20230725" in key or "20230727" in key or "20230726" in key]
filenames

['groundbased_lidar.o3_ccny001_hires_new.york.ny_20230726t131528z_20230727t001119z_001.h5',
 'groundbased_lidar.o3_ccny001_hires_new.york.ny_20230726t131628z_20230727t001119z_001.h5',
 'groundbased_lidar.o3_ccny001_hires_new.york.ny_20230727t143727z_20230727t210523z_001.h5',
 'groundbased_lidar.o3_nasa.gsfc003_hires.glass.1.1_oldfield.ny_20230725t003007z_20230725t012317z_001.h5',
 'groundbased_lidar.o3_nasa.gsfc003_hires.glass.1.1_oldfield.ny_20230726t121404z_20230727t000058z_001.h5',
 'groundbased_lidar.o3_nasa.gsfc003_hires.glass.1.1_oldfield.ny_20230727t000018z_20230727t235956z_001.h5',
 'groundbased_lidar.o3_nasa.gsfc003_hires_oldfield.ny_20230725t000000z_20230726t000000z_001.h5',
 'groundbased_lidar.o3_nasa.gsfc003_hires_oldfield.ny_20230726t000000z_20230727t000000z_001.h5',
 'groundbased_lidar.o3_nasa.gsfc003_hires_oldfield.ny_20230727t000000z_20230728t000000z_001.h5',
 'groundbased_lidar.o3_noaa.csl001_hires_guilford.ycfs.ct_20230725t132403z_20230725t200403z_002.h5',
 'groundbas

In [11]:
run = True

if run==True:
    from collections import defaultdict

    grouped = defaultdict(list)

    for key in comparison.keys():
        grouped[key[0]].append(key)
    
    temp_sondes = {}; temp_tolnet = {}
    for lidar_file, matching_keys in grouped.items():
        if "glass" in lidar_file:
            instrument_name = instrument_names[lidar_file.split('.')[2]] + " GLASS"
        else:
            instrument_name = instrument_names[lidar_file.split('.')[2]]

        if instrument_name not in temp_sondes.keys():
            temp_sondes[instrument_name] = []
        if instrument_name not in temp_tolnet.keys():
            temp_tolnet[instrument_name] = []

        temp_sonde = []
        for k in matching_keys:
            temp_sonde.append(comparison[k]["left_df"])

        temp_sondes[instrument_name].append(pd.concat(temp_sonde))
        temp_tolnet[instrument_name].append(tolnet[lidar_file].drop(columns="geometry"))

    for instrument_name in temp_tolnet.keys():
        temp_sondes[instrument_name] = pd.concat(temp_sondes[instrument_name])
        temp_tolnet[instrument_name] = pd.concat(temp_tolnet[instrument_name])

 

In [21]:
start_date = pd.Timestamp("2023-07-26 12:00")
end_date = pd.Timestamp("2023-07-27 12:00")

figures_dir = fig_dir / "curtains_with_sondes"
figures_dir.mkdir(exist_ok=True)

for instrument_name in temp_tolnet.keys():
    if "glass" in instrument_name.lower():
        title = f"{instrument_name} [{start_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')}]"
    else:
        title = f"{instrument_name} In-House [{end_date.strftime('%Y-%m-%d')} - {end_date.strftime('%Y-%m-%d')}]"

    temp_tolnet[instrument_name] = temp_tolnet[instrument_name].resample("30min").mean()

    sonde = {
        "X": temp_sondes[instrument_name].index.get_level_values('timestamp'),
        "Y": temp_sondes[instrument_name]["Altitude_km"],
        "C": temp_sondes[instrument_name]["Ozone_ppbv"],
        "vline_params": {
            "colors": 'black', 
            "linewidth": 1.5, 
            "alpha": 0.5, 
            "zorder": 3,
            "linestyle": 'dashed',
            "skip": 15
            }
        }

    lidar = {
        "X": temp_tolnet[instrument_name].index,
        "Y": temp_tolnet[instrument_name].columns.astype(float) / 1000,
        "C": temp_tolnet[instrument_name].values.T * 1000
        }

    params = {
        "fig.colorbar": {
            "label": "Ozone Mixing Ratio (ppbv)"
            },
        "ax.set_xlim": [start_date, end_date],
        "ax.set_ylim": [0, 5],
        "ax.set_title": title,
        "fig.savefig": {
            "fname":  figures_dir / f"{title.replace('-',' ').replace(' ', '_')}.png",
            "format": "png",
            "dpi": 600,
            "transparent": True
            }
        }

    makePlots.plot_curtain(lidar, sonde, **params)